# 08 — Temporal Cross-Validation

Standard k-fold CV shuffles cases randomly, allowing future cases to appear in the training set.
This produces an optimistic AUC estimate that would not hold in production.

**Temporal CV** trains strictly on cases that opened *before* the test window — matching real deployment:
a model trained today predicts on cases opened tomorrow.

This notebook answers:
1. How much does standard k-fold overestimate performance vs temporal CV?
2. Does prefix-model accuracy hold across time (stable deployment)?
3. Is there **data drift** — do feature distributions shift quarter to quarter?
4. Is there **concept drift** — does the relationship between features and outcome change over time?

**Folds (expanding window):**
| Test window | Training data |
|---|---|
| Q1 2018 | Pre-2018 (all 2017) |
| Q2 2018 | Pre-Q2 2018 |
| Q3 2018 | Pre-Q3 2018 |
| Q4 2018 | Pre-Q4 2018 |

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection  import StratifiedKFold, cross_val_score
from sklearn.preprocessing    import LabelEncoder
from sklearn.impute            import SimpleImputer
from sklearn.metrics           import roc_auc_score
from src.load_event_log import load_xes_log, convert_to_dataframe

DATA_PATH   = Path('../data/raw/PermitLog.xes')
FIGURES_OUT = Path('../outputs/figures')
TABLES_OUT  = Path('../outputs/tables')
FIGURES_OUT.mkdir(parents=True, exist_ok=True)
TABLES_OUT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
LONG_THRESHOLD_DAYS = 101.1   # overall p67 from Notebook 06

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load event log and compute case start dates

In [2]:
df = convert_to_dataframe(load_xes_log(DATA_PATH, legacy=True))
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], utc=True)
df = df.sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)

case_col = 'case:concept:name'
act_col  = 'concept:name'
time_col = 'time:timestamp'

# Case start date (first event timestamp, tz-naive for easier comparison)
case_start = (
    df.groupby(case_col)[time_col].min()
    .rename('case_start')
    .reset_index()
    .rename(columns={case_col: 'case_id'})
)
case_start['case_start'] = case_start['case_start'].dt.tz_localize(None)

# Case duration and target label (using fixed overall threshold)
case_end = (
    df.groupby(case_col)[time_col].max()
    .rename('case_end')
    .reset_index()
    .rename(columns={case_col: 'case_id'})
)
case_end['case_end'] = case_end['case_end'].dt.tz_localize(None)

case_meta = case_start.merge(case_end, on='case_id')
case_meta['duration_days'] = (case_meta['case_end'] - case_meta['case_start']).dt.total_seconds() / 86400
case_meta['is_long'] = (case_meta['duration_days'] >= LONG_THRESHOLD_DAYS).astype(int)
case_meta['quarter'] = pd.PeriodIndex(case_meta['case_start'], freq='Q')

print(f'Cases: {len(case_meta):,}  |  Long-duration rate: {case_meta["is_long"].mean():.3f}')
print(f'Date range: {case_meta["case_start"].min().date()} → {case_meta["case_start"].max().date()}')
print()
print('Cases per quarter:')
qsum = case_meta.groupby('quarter').agg(n=('case_id','count'), pct_long=('is_long','mean')).round(3)
print(qsum.to_string())

parsing log, completed traces ::   0%|          | 0/7065 [00:00<?, ?it/s]

parsing log, completed traces ::  15%|█▌        | 1071/7065 [00:00<00:00, 10708.07it/s]

parsing log, completed traces ::  30%|███       | 2142/7065 [00:00<00:00, 9895.94it/s] 

parsing log, completed traces ::  44%|████▍     | 3136/7065 [00:00<00:00, 9345.31it/s]

parsing log, completed traces ::  58%|█████▊    | 4075/7065 [00:00<00:00, 9240.75it/s]

parsing log, completed traces ::  71%|███████   | 5001/7065 [00:00<00:00, 9203.61it/s]

parsing log, completed traces ::  84%|████████▍ | 5923/7065 [00:00<00:00, 9188.83it/s]

parsing log, completed traces :: 100%|██████████| 7065/7065 [00:00<00:00, 9657.04it/s]

Cases: 7,065  |  Long-duration rate: 0.333
Date range: 2016-10-05 → 2018-12-30

Cases per quarter:
            n  pct_long
quarter                
2016Q4      4     0.500
2017Q1    389     0.386
2017Q2    441     0.420
2017Q3    338     0.308
2017Q4    295     0.319
2018Q1   1596     0.380
2018Q2   1713     0.368
2018Q3   1227     0.249
2018Q4   1062     0.263


## 2. Case volume and long-duration rate over time

In [3]:
monthly = case_meta.copy()
monthly['month'] = monthly['case_start'].dt.to_period('M')
mon_stats = monthly.groupby('month').agg(
    n=('case_id','count'),
    pct_long=('is_long','mean')
).reset_index()
mon_stats['month_dt'] = mon_stats['month'].dt.to_timestamp()

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.bar(mon_stats['month_dt'], mon_stats['n'], width=20,
        color='steelblue', alpha=0.6, label='Case volume')
ax2.plot(mon_stats['month_dt'], mon_stats['pct_long'] * 100,
         'o-', color='#d7191c', linewidth=2, markersize=5, label='% Long-duration')

# Shade fold test windows
fold_windows = [
    ('Q1 2018', '2018-01-01', '2018-04-01', '#ffffb2'),
    ('Q2 2018', '2018-04-01', '2018-07-01', '#fecc5c'),
    ('Q3 2018', '2018-07-01', '2018-10-01', '#fd8d3c'),
    ('Q4 2018', '2018-10-01', '2019-01-01', '#e31a1c'),
]
for label, t0, t1, color in fold_windows:
    ax1.axvspan(pd.Timestamp(t0), pd.Timestamp(t1), alpha=0.15,
                color=color, label=f'Test: {label}')
    ax1.text(pd.Timestamp(t0) + pd.Timedelta(days=15), ax1.get_ylim()[1] * 0.88,
             label, fontsize=7, color='#333333')

ax1.set_xlabel('Month')
ax1.set_ylabel('Cases opened', color='steelblue')
ax2.set_ylabel('% Long-duration cases', color='#d7191c')
ax1.set_title('Monthly Case Volume and Long-Duration Rate — with Temporal CV Fold Windows', fontsize=11)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1[:2] + lines2, labels1[:2] + labels2, loc='upper left', fontsize=8)
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'temporal_volume_folds.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved temporal_volume_folds.png')

Saved temporal_volume_folds.png


## 3. Prefix feature engineering

Same function as Notebook 07, reproduced here for self-containment.

In [4]:
# Case-level attributes
case_attrs = (
    df.drop_duplicates(case_col)
    [[case_col, 'case:OrganizationalEntity', 'case:RequestedBudget']]
    .rename(columns={case_col: 'case_id'})
    .copy()
)
case_attrs['case:RequestedBudget'] = pd.to_numeric(case_attrs['case:RequestedBudget'], errors='coerce')
le_org = LabelEncoder()
case_attrs['org_encoded'] = le_org.fit_transform(case_attrs['case:OrganizationalEntity'].astype(str))

TOP_ACTS = df[act_col].value_counts().head(15).index.tolist()

def make_prefix_features(df, k):
    prefix = (
        df.sort_values([case_col, time_col])
        .groupby(case_col, group_keys=False)
        .head(k)
        .copy()
    )
    first_ts = df.groupby(case_col)[time_col].min()
    kth_ts   = prefix.groupby(case_col)[time_col].max()
    elapsed  = ((kth_ts - first_ts).dt.total_seconds() / 86400).rename('elapsed_days')

    n_events = prefix.groupby(case_col).size().rename('n_events_prefix')
    n_rej = (prefix[prefix[act_col].str.contains('REJECTED', na=False)]
             .groupby(case_col).size().rename('n_rejections')
             .reindex(df[case_col].unique(), fill_value=0))
    n_rem = (prefix[prefix[act_col] == 'Send Reminder']
             .groupby(case_col).size().rename('n_reminders')
             .reindex(df[case_col].unique(), fill_value=0))
    n_app = (prefix[prefix[act_col].str.contains('APPROVED', na=False)]
             .groupby(case_col).size().rename('n_approvals')
             .reindex(df[case_col].unique(), fill_value=0))

    first_act = (
        prefix.sort_values([case_col, time_col])
        .groupby(case_col)[act_col].first()
        .rename('first_act')
    )
    le_act = LabelEncoder()
    first_act_enc = le_act.fit_transform(first_act.astype(str))

    flags = {}
    for act in TOP_ACTS:
        col_name = 'has_' + act.replace(' ', '_').replace(':', '_')[:30]
        flags[col_name] = (
            (prefix[prefix[act_col] == act].groupby(case_col).size() > 0)
            .astype(int)
        )

    result = pd.DataFrame({'case_id': df[case_col].unique()})
    for s in [elapsed, n_events, n_rej, n_rem, n_app]:
        result = result.merge(
            s.reset_index().rename(columns={case_col: 'case_id'}),
            on='case_id', how='left'
        )
    result['first_act_enc'] = (
        result.merge(
            pd.Series(first_act_enc, index=first_act.index, name='fa')
            .reset_index().rename(columns={case_col: 'case_id'}),
            on='case_id', how='left'
        )['fa'].fillna(-1).astype(int).values
    )
    for name, s in flags.items():
        result = result.merge(
            s.reset_index().rename(columns={case_col: 'case_id', 0: name}),
            on='case_id', how='left'
        )
        result[name] = result[name].fillna(0).astype(int)

    result = result.merge(case_attrs[['case_id','org_encoded','case:RequestedBudget']],
                          on='case_id', how='left')
    result = result.merge(case_meta[['case_id','case_start','is_long','duration_days']],
                          on='case_id', how='left')
    return result

# Pre-compute for k=1,3,5,8 (store for re-use)
prefix_data = {}
PREFIX_LENGTHS = [1, 3, 5, 8]
for k in PREFIX_LENGTHS:
    prefix_data[k] = make_prefix_features(df, k)
    print(f'k={k}  rows={len(prefix_data[k])}')

FEAT_DROP = ['case_id', 'case_start', 'is_long', 'duration_days']
FEAT_COLS  = [c for c in prefix_data[1].columns if c not in FEAT_DROP]
print(f'\nFeatures ({len(FEAT_COLS)}): {FEAT_COLS}')

k=1  rows=7065


k=3  rows=7065


k=5  rows=7065


k=8  rows=7065

Features (23): ['elapsed_days', 'n_events_prefix', 'n_rejections', 'n_reminders', 'n_approvals', 'first_act_enc', 'has_Declaration_SUBMITTED_by_EMPLO', 'has_Payment_Handled', 'has_Request_Payment', 'has_Permit_SUBMITTED_by_EMPLOYEE', 'has_Start_trip', 'has_End_trip', 'has_Permit_FINAL_APPROVED_by_SUPER', 'has_Permit_APPROVED_by_ADMINISTRAT', 'has_Declaration_FINAL_APPROVED_by_', 'has_Declaration_APPROVED_by_ADMINI', 'has_Send_Reminder', 'has_Permit_APPROVED_by_BUDGET_OWNE', 'has_Request_For_Payment_SUBMITTED_', 'has_Request_For_Payment_FINAL_APPR', 'has_Declaration_APPROVED_by_BUDGET', 'org_encoded', 'case:RequestedBudget']


## 4. Define temporal folds

In [5]:
TEMPORAL_FOLDS = [
    ('Q1 2018', pd.Timestamp('2018-01-01'), pd.Timestamp('2018-04-01')),
    ('Q2 2018', pd.Timestamp('2018-04-01'), pd.Timestamp('2018-07-01')),
    ('Q3 2018', pd.Timestamp('2018-07-01'), pd.Timestamp('2018-10-01')),
    ('Q4 2018', pd.Timestamp('2018-10-01'), pd.Timestamp('2019-01-01')),
]

for fold_name, t0, t1 in TEMPORAL_FOLDS:
    n_train = (case_meta['case_start'] < t0).sum()
    n_test  = ((case_meta['case_start'] >= t0) & (case_meta['case_start'] < t1)).sum()
    pct_long_train = case_meta.loc[case_meta['case_start'] < t0, 'is_long'].mean()
    pct_long_test  = case_meta.loc[
        (case_meta['case_start'] >= t0) & (case_meta['case_start'] < t1), 'is_long'].mean()
    print(f"{fold_name}  train={n_train:4d} (long={pct_long_train:.2f})  "
          f"test={n_test:4d} (long={pct_long_test:.2f})")

Q1 2018  train=1467 (long=0.36)  test=1596 (long=0.38)
Q2 2018  train=3063 (long=0.37)  test=1713 (long=0.37)
Q3 2018  train=4776 (long=0.37)  test=1227 (long=0.25)
Q4 2018  train=6003 (long=0.35)  test=1062 (long=0.26)


## 5. Temporal CV — AUC per fold per prefix length

In [6]:
def fit_eval_xgb(X_train, y_train, X_test, y_test):
    imp = SimpleImputer(strategy='median')
    Xtr = pd.DataFrame(imp.fit_transform(X_train), columns=X_train.columns)
    Xte = pd.DataFrame(imp.transform(X_test),       columns=X_test.columns)
    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
    )
    model.fit(Xtr, y_train)
    y_proba = model.predict_proba(Xte)[:, 1]
    return model, Xtr, Xte, roc_auc_score(y_test, y_proba)

temporal_results = []

for k in PREFIX_LENGTHS:
    pf = prefix_data[k]
    feat_cols = [c for c in FEAT_COLS if c in pf.columns]

    for fold_name, t0, t1 in TEMPORAL_FOLDS:
        train_mask = pf['case_start'] < t0
        test_mask  = (pf['case_start'] >= t0) & (pf['case_start'] < t1)

        X_tr, y_tr = pf.loc[train_mask, feat_cols], pf.loc[train_mask, 'is_long']
        X_te, y_te = pf.loc[test_mask,  feat_cols], pf.loc[test_mask,  'is_long']

        if len(X_tr) < 50 or len(X_te) < 20 or y_te.nunique() < 2:
            continue

        _, _, _, auc = fit_eval_xgb(X_tr, y_tr.values, X_te, y_te.values)

        temporal_results.append({
            'k': k, 'fold': fold_name,
            'n_train': train_mask.sum(), 'n_test': test_mask.sum(),
            'test_auc': auc,
        })
        print(f'k={k}  {fold_name}  n_train={train_mask.sum():4d}  '
              f'n_test={test_mask.sum():4d}  AUC={auc:.4f}')

temporal_df = pd.DataFrame(temporal_results)
temporal_df.to_csv(TABLES_OUT / 'temporal_cv_results.csv', index=False)
print('\nSaved temporal_cv_results.csv')

k=1  Q1 2018  n_train=1467  n_test=1596  AUC=0.5644


k=1  Q2 2018  n_train=3063  n_test=1713  AUC=0.6512


k=1  Q3 2018  n_train=4776  n_test=1227  AUC=0.6543


k=1  Q4 2018  n_train=6003  n_test=1062  AUC=0.6480


k=3  Q1 2018  n_train=1467  n_test=1596  AUC=0.6542


...

k=5  Q4 2018  n_train=6003  n_test=1062  AUC=0.7514


k=8  Q1 2018  n_train=1467  n_test=1596  AUC=0.9472


k=8  Q2 2018  n_train=3063  n_test=1713  AUC=0.9643


k=8  Q3 2018  n_train=4776  n_test=1227  AUC=0.9646


k=8  Q4 2018  n_train=6003  n_test=1062  AUC=0.9906

Saved temporal_cv_results.csv


## 6. Standard k-fold vs temporal CV — bias estimate

In [7]:
kfold_results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for k in PREFIX_LENGTHS:
    pf = prefix_data[k]
    feat_cols = [c for c in FEAT_COLS if c in pf.columns]
    imp = SimpleImputer(strategy='median')
    X_all = pd.DataFrame(imp.fit_transform(pf[feat_cols]), columns=feat_cols)
    y_all = pf['is_long'].values

    model_k = xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
    )
    cv_aucs = cross_val_score(model_k, X_all, y_all, cv=skf, scoring='roc_auc')
    kfold_results.append({'k': k, 'kfold_auc_mean': cv_aucs.mean(), 'kfold_auc_std': cv_aucs.std()})
    print(f'k={k}  5-fold CV AUC = {cv_aucs.mean():.4f} ± {cv_aucs.std():.4f}')

kfold_df = pd.DataFrame(kfold_results)

# Compare: mean temporal AUC vs k-fold AUC
temporal_mean = temporal_df.groupby('k')['test_auc'].mean().reset_index().rename(columns={'test_auc':'temporal_auc_mean'})
comparison = kfold_df.merge(temporal_mean, on='k')
comparison['bias'] = comparison['kfold_auc_mean'] - comparison['temporal_auc_mean']
print('\n=== k-fold vs Temporal CV ===')
print(comparison[['k','kfold_auc_mean','temporal_auc_mean','bias']].round(4).to_string(index=False))
comparison.to_csv(TABLES_OUT / 'temporal_vs_kfold.csv', index=False)

k=1  5-fold CV AUC = 0.6556 ± 0.0107


k=3  5-fold CV AUC = 0.6964 ± 0.0130


k=5  5-fold CV AUC = 0.8298 ± 0.0063


k=8  5-fold CV AUC = 0.9747 ± 0.0021

=== k-fold vs Temporal CV ===
 k  kfold_auc_mean  temporal_auc_mean   bias
 1          0.6556             0.6295 0.0261
 3          0.6964             0.6687 0.0276
 5          0.8298             0.7822 0.0476
 8          0.9747             0.9667 0.0080


## 7. Visualise: AUC heatmap — fold × prefix length

In [8]:
pivot = temporal_df.pivot(index='fold', columns='k', values='test_auc')
fold_order = [f for f, _, _ in TEMPORAL_FOLDS]
pivot = pivot.reindex(fold_order)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0.5, vmax=1.0, aspect='auto')
plt.colorbar(im, ax=ax, label='Test ROC-AUC')

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'k={k}' for k in pivot.columns], fontsize=10)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=10)
ax.set_xlabel('Prefix length')
ax.set_title('Temporal CV — AUC by Fold and Prefix Length', fontsize=11)

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=10, fontweight='bold',
                    color='black' if val > 0.65 else 'white')

plt.tight_layout()
fig.savefig(FIGURES_OUT / 'temporal_auc_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved temporal_auc_heatmap.png')

Saved temporal_auc_heatmap.png


## 8. k-fold vs temporal CV bias chart

In [9]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(PREFIX_LENGTHS))
w = 0.35

bars1 = ax.bar(x - w/2, comparison['kfold_auc_mean'], w,
               label='Standard 5-fold CV (shuffled)', color='#2c7bb6', alpha=0.85,
               yerr=comparison['kfold_auc_std'], capsize=4, error_kw={'linewidth':1.5})
bars2 = ax.bar(x + w/2, comparison['temporal_auc_mean'], w,
               label='Temporal CV (expanding window, mean over 4 folds)',
               color='#d7191c', alpha=0.85)

# Bias annotations
for xi, (_, row) in zip(x, comparison.iterrows()):
    bias = row['kfold_auc_mean'] - row['temporal_auc_mean']
    ax.annotate(f'bias\n+{bias:.3f}',
                xy=(xi, max(row['kfold_auc_mean'], row['temporal_auc_mean']) + 0.01),
                ha='center', fontsize=8, color='#555555')

ax.set_xticks(x)
ax.set_xticklabels([f'k={k}' for k in PREFIX_LENGTHS], fontsize=11)
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0.5, 1.12)
ax.set_title('Optimism Bias: Standard k-fold vs Temporal CV\n'
             '(Gap = information leaked from future cases in shuffled CV)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'temporal_vs_kfold_bias.png', dpi=150)
plt.close()
print('Saved temporal_vs_kfold_bias.png')

Saved temporal_vs_kfold_bias.png


## 9. Data drift — key feature distributions by quarter

In [10]:
# Use k=5 prefix for drift analysis (operationally relevant)
pf5 = prefix_data[5].copy()
pf5['quarter'] = pd.PeriodIndex(pf5['case_start'], freq='Q').astype(str)

drift_features = ['elapsed_days', 'n_approvals', 'n_reminders', 'case:RequestedBudget']

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
quarters = sorted(pf5['quarter'].dropna().unique())
palette  = plt.cm.viridis(np.linspace(0.1, 0.9, len(quarters)))

for ax, feat in zip(axes.flat, drift_features):
    for q, color in zip(quarters, palette):
        vals = pf5.loc[pf5['quarter'] == q, feat].dropna()
        if len(vals) < 10:
            continue
        # Clip extreme tails for display
        clip = vals.quantile(0.99)
        vals = vals.clip(upper=clip)
        ax.hist(vals, bins=25, alpha=0.5, color=color,
                label=q, density=True, histtype='step', linewidth=1.5)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')

# Shared legend
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Quarter', bbox_to_anchor=(1.01, 0.5),
           loc='center left', fontsize=8)
plt.suptitle('Feature Distribution Drift by Quarter — k=5 Prefix', fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'temporal_feature_drift.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved temporal_feature_drift.png')

# Drift table: median per quarter
drift_table = pf5.groupby('quarter')[drift_features].median().round(2)
drift_table.to_csv(TABLES_OUT / 'feature_drift_by_quarter.csv')
print(drift_table.to_string())

Saved temporal_feature_drift.png
         elapsed_days  n_approvals  n_reminders  case:RequestedBudget
quarter                                                              
2016Q4          82.12          1.0          0.0                 46.70
2017Q1          39.08          2.0          0.0                736.72
2017Q2          32.97          2.0          0.0                944.69
2017Q3          28.54          2.0          0.0                921.86
2017Q4          30.91          2.0          0.0                517.45
2018Q1          18.01          2.0          0.0                934.57
2018Q2          17.21          2.0          0.0                855.57
2018Q3          17.59          2.0          0.0                893.79
2018Q4          15.99          2.0          0.0                623.49


## 10. Concept drift — SHAP importance: early cohort vs late cohort

Train the k=5 model twice:
- **Early:** cases opening in 2017 (all quarters)
- **Late:** cases opening in H2 2018 (Q3+Q4)

Compare SHAP feature importances to check if what predicts long-duration changed.

In [11]:
pf5 = prefix_data[5].copy()
feat_cols_5 = [c for c in FEAT_COLS if c in pf5.columns]

early_mask = pf5['case_start'] < pd.Timestamp('2018-01-01')
late_mask  = pf5['case_start'] >= pd.Timestamp('2018-07-01')

print(f'Early cohort (2017): {early_mask.sum()} cases')
print(f'Late cohort  (H2 2018): {late_mask.sum()} cases')

def train_and_shap(mask, label):
    sub = pf5.loc[mask].copy()
    imp = SimpleImputer(strategy='median')
    X = pd.DataFrame(imp.fit_transform(sub[feat_cols_5]), columns=feat_cols_5)
    y = sub['is_long'].values
    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
    )
    model.fit(X, y)
    explainer = shap.TreeExplainer(model)
    sv = explainer(X)
    mean_abs = np.abs(sv.values).mean(axis=0)
    importance = pd.Series(
        (mean_abs / mean_abs.sum() * 100).round(2),
        index=feat_cols_5, name=label
    ).sort_values(ascending=False)
    print(f'\n{label} — top 5:')
    print(importance.head(5).to_string())
    return importance

shap_early = train_and_shap(early_mask, 'Early (2017)')
shap_late  = train_and_shap(late_mask,  'Late (H2 2018)')

Early cohort (2017): 1467 cases
Late cohort  (H2 2018): 2289 cases



Early (2017) — top 5:
elapsed_days                          53.560001
has_Declaration_SUBMITTED_by_EMPLO    13.520000
case:RequestedBudget                   9.480000
n_approvals                            6.910000
has_Start_trip                         4.180000



Late (H2 2018) — top 5:
elapsed_days                          35.18
case:RequestedBudget                  27.09
has_Start_trip                        11.06
org_encoded                           10.09
has_Request_For_Payment_SUBMITTED_     5.06


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (imp, color, title) in zip(axes, [
    (shap_early, '#2c7bb6', 'Early cohort — 2017\n(k=5 prefix, SHAP)'),
    (shap_late,  '#d7191c', 'Late cohort — H2 2018\n(k=5 prefix, SHAP)'),
]):
    top = imp.head(10)
    ax.barh(top.index[::-1], top.values[::-1], color=color, alpha=0.85)
    ax.set_xlabel('Mean |SHAP| (% of total)', fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Concept Drift Check — SHAP Importances: Early vs Late Cohort', fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'temporal_concept_drift.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved temporal_concept_drift.png')

# Rank correlation between early and late importance
from scipy.stats import spearmanr
common = shap_early.index.intersection(shap_late.index)
rho, pval = spearmanr(shap_early[common].values, shap_late[common].values)
print(f'\nSpearman rank correlation of SHAP importances: ρ={rho:.3f}  p={pval:.4f}')
print('(ρ→1 means same features matter in both periods — no concept drift)')

Saved temporal_concept_drift.png

Spearman rank correlation of SHAP importances: ρ=0.788  p=0.0000
(ρ→1 means same features matter in both periods — no concept drift)


## 11. Training size sensitivity — minimum data needed for stable prediction

In [13]:
pf5 = prefix_data[5].copy()
feat_cols_5 = [c for c in FEAT_COLS if c in pf5.columns]

pf5_sorted = pf5.sort_values('case_start').reset_index(drop=True)
test_mask  = (pf5_sorted['case_start'] >= pd.Timestamp('2018-10-01')) & \
             (pf5_sorted['case_start'] <  pd.Timestamp('2019-01-01'))
X_test_fixed = pf5_sorted.loc[test_mask, feat_cols_5]
y_test_fixed = pf5_sorted.loc[test_mask, 'is_long'].values

train_pool = pf5_sorted[~test_mask].reset_index(drop=True)
train_sizes = [100, 200, 400, 700, 1000, 1500, 2000, 3000, 4000, len(train_pool)]

size_results = []
for n in train_sizes:
    if n > len(train_pool):
        n = len(train_pool)
    subset = train_pool.iloc[:n]   # earliest n cases
    y_tr = subset['is_long'].values
    if len(np.unique(y_tr)) < 2:
        continue
    imp = SimpleImputer(strategy='median')
    X_tr = pd.DataFrame(imp.fit_transform(subset[feat_cols_5]), columns=feat_cols_5)
    X_te = pd.DataFrame(imp.transform(X_test_fixed),           columns=feat_cols_5)

    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
    )
    model.fit(X_tr, y_tr)
    auc = roc_auc_score(y_test_fixed, model.predict_proba(X_te)[:, 1])
    size_results.append({'n_train': n, 'auc': auc})
    print(f'n_train={n:5d}  AUC={auc:.4f}')

size_df = pd.DataFrame(size_results)
size_df.to_csv(TABLES_OUT / 'training_size_sensitivity.csv', index=False)

n_train=  100  AUC=0.5413


n_train=  200  AUC=0.5555


n_train=  400  AUC=0.6400


n_train=  700  AUC=0.6374


n_train= 1000  AUC=0.6780


n_train= 1500  AUC=0.6957


n_train= 2000  AUC=0.7131


n_train= 3000  AUC=0.7249


n_train= 4000  AUC=0.7452


n_train= 6003  AUC=0.7526


In [14]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(size_df['n_train'], size_df['auc'], 'o-',
        color='#2c7bb6', linewidth=2, markersize=7)

for _, row in size_df.iterrows():
    ax.annotate(f"{row['auc']:.3f}",
                (row['n_train'], row['auc']),
                textcoords='offset points', xytext=(0, 8), fontsize=8, ha='center')

ax.set_xlabel('Number of training cases (earliest N, chronological)', fontsize=11)
ax.set_ylabel('ROC-AUC on Q4 2018 test set', fontsize=11)
ax.set_title('Training Size Sensitivity — k=5 Prefix Model\n'
             'Test fixed at Q4 2018 cases', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 1.05)
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'temporal_training_size.png', dpi=150)
plt.close()
print('Saved temporal_training_size.png')

Saved temporal_training_size.png


## 12. Summary

In [15]:
print('=== NOTEBOOK 08 — TEMPORAL CROSS-VALIDATION SUMMARY ===')
print()
print('Standard k-fold vs Temporal CV (mean over 4 quarterly folds):')
print(comparison[['k','kfold_auc_mean','temporal_auc_mean','bias']]
      .rename(columns={'kfold_auc_mean':'kfold_AUC','temporal_auc_mean':'temporal_AUC'})
      .round(4).to_string(index=False))
print()
print('Temporal CV AUC by fold:')
print(pivot.round(3).to_string())
print()
from scipy.stats import spearmanr
common = shap_early.index.intersection(shap_late.index)
rho, _ = spearmanr(shap_early[common].values, shap_late[common].values)
print(f'Concept drift check — SHAP rank correlation (early vs late): ρ = {rho:.3f}')
if rho > 0.8:
    print('  → No significant concept drift: same features matter across time')
else:
    print('  → Possible concept drift detected: feature rankings differ materially')
print()
stable_n = size_df[size_df['auc'] >= size_df['auc'].max() * 0.95]['n_train'].min()
print(f'Minimum training size for 95% of peak AUC: ~{stable_n} cases')

=== NOTEBOOK 08 — TEMPORAL CROSS-VALIDATION SUMMARY ===

Standard k-fold vs Temporal CV (mean over 4 quarterly folds):
 k  kfold_AUC  temporal_AUC   bias
 1     0.6556        0.6295 0.0261
 3     0.6964        0.6687 0.0276
 5     0.8298        0.7822 0.0476
 8     0.9747        0.9667 0.0080

Temporal CV AUC by fold:
k            1      3      5      8
fold                               
Q1 2018  0.564  0.654  0.791  0.947
Q2 2018  0.651  0.679  0.813  0.964
Q3 2018  0.654  0.678  0.774  0.965
Q4 2018  0.648  0.664  0.751  0.991

Concept drift check — SHAP rank correlation (early vs late): ρ = 0.788
  → Possible concept drift detected: feature rankings differ materially

Minimum training size for 95% of peak AUC: ~3000 cases
